# 56. Models (final)
## Contents
- Prerequisites
- Results
    - Logistic Regression
    - Random Forest
    - Support Vector Machine
- Final Model
---------------------------------------------------------
## Prerequisites

In [2]:
import time
import os
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from tqdm import tqdm
import joblib
import warnings
from sklearn.model_selection import cross_val_score
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
file_path = 'data/'
parties = np.load('00_parties.npy')
options = ['imbalanced','oversampling','undersampling','balancedsampling']
rs = 1

------------------------------------------------
<br>
<br>
<br>


## Results
### Logistic Regression
--------------------------------------------

In [45]:
file_url  = file_path + '53_data_tuned_logistic_regression.csv' 
result_lr = pd.read_csv(file_url)
columns = ['option', 'C', 'max_iter']
result_lr = result_lr.drop(['option'], axis=1)
result_lr.columns = result_lr.columns.map(lambda x: 'lr_' + x if x in columns else x)

avg_best_lr = result_lr['lr_score'].mean()                                             # define average best values
print("Average accuracy: ", avg_best_lr)                                               # print average best values

display(result_lr)

Average accuracy:  0.7456245106521546


,party,lr_C,lr_max_iter,lr_score
0,50PLUS,10.000,1000,0.802179
1,BBB,0.001,100,0.837973
2,BIJ1,1000.000,10000,0.782581
3,BVNL,0.100,100,0.680046
4,CDA,0.001,100,0.676357
5,CU,100.000,1000,0.687045
6,D66,0.001,100,0.682744
7,DENK,100.000,100,0.834264
8,FVD,1.000,1000,0.695194
9,GLPvdA,100.000,1000,0.821195


------------------------------------
### Random Forest

In [46]:
file_url  = file_path + '54_data_tuned_random_forest.csv' 
result_rf = pd.read_csv(file_url)
columns = ['option', 'n_est', 'max_depth','min_split']
result_rf = result_rf.drop(['option'], axis=1)
result_rf.columns = result_rf.columns.map(lambda x: 'rf_' + x if x in columns else x)

avg_best_rf = result_rf['rf_score'].mean()                                             # define average best values
print("Average accuracy: ", avg_best_rf)                                               # print average best values

display(result_rf)

Average accuracy:  0.7543124566433861


,party,rf_n_est,rf_max_depth,rf_min_split,rf_score
0,50PLUS,500,50,2,0.807130
1,BBB,250,25,2,0.841314
2,BIJ1,250,50,2,0.790968
3,BVNL,500,25,2,0.692783
4,CDA,250,25,5,0.676357
5,CU,500,25,5,0.691412
6,D66,500,50,5,0.695104
7,DENK,500,50,2,0.840087
8,FVD,500,25,2,0.714566
9,GLPvdA,250,50,5,0.822889


-------------------------
### Support Vector Machine

In [47]:
file_url  = file_path + '55_data_tuned_support_vector_machine.csv' 
result_svm = pd.read_csv(file_url)
columns = ['option', 'C', 'kernel','gamma']
result_svm = result_svm.drop(['option'], axis=1)
result_svm.columns = result_svm.columns.map(lambda x: 'svm_' + x if x in columns else x)

avg_best_svm = result_svm['svm_score'].mean()                                          # define average best values
print("Average accuracy: ", avg_best_svm)                                              # print average best values

display(result_svm)

Average accuracy:  0.6932356457353035


,party,svm_C,svm_kernel,svm_gamma,svm_score
0,50PLUS,0.1,rbf,scale,0.790295
1,BBB,0.1,rbf,scale,0.830735
2,BIJ1,0.1,rbf,scale,0.758065
3,BVNL,0.1,rbf,scale,0.632960
4,CDA,10.0,rbf,scale,0.544816
5,CU,10.0,rbf,scale,0.571082
6,D66,10.0,rbf,scale,0.557925
7,DENK,0.1,rbf,scale,0.824800
8,FVD,0.1,rbf,scale,0.605934
9,GLPvdA,0.1,rbf,scale,0.801839


--------------------------

In [48]:
result_all = pd.merge(result_lr, result_rf, on='party', how='inner')
result_all = pd.merge(result_all, result_svm, on='party', how='inner')
display(result_all)

,party,lr_C,lr_max_iter,lr_score,rf_n_est,rf_max_depth,rf_min_split,rf_score,svm_C,svm_kernel,svm_gamma,svm_score
0,50PLUS,10.000,1000,0.802179,500,50,2,0.807130,0.1,rbf,scale,0.790295
1,BBB,0.001,100,0.837973,250,25,2,0.841314,0.1,rbf,scale,0.830735
2,BIJ1,1000.000,10000,0.782581,250,50,2,0.790968,0.1,rbf,scale,0.758065
3,BVNL,0.100,100,0.680046,500,25,2,0.692783,0.1,rbf,scale,0.632960
4,CDA,0.001,100,0.676357,250,25,5,0.676357,10.0,rbf,scale,0.544816
5,CU,100.000,1000,0.687045,500,25,5,0.691412,10.0,rbf,scale,0.571082
6,D66,0.001,100,0.682744,500,50,5,0.695104,10.0,rbf,scale,0.557925
7,DENK,100.000,100,0.834264,500,50,2,0.840087,0.1,rbf,scale,0.824800
8,FVD,1.000,1000,0.695194,500,25,2,0.714566,0.1,rbf,scale,0.605934
9,GLPvdA,100.000,1000,0.821195,250,50,5,0.822889,0.1,rbf,scale,0.801839


In [49]:
result_best = result_all.copy()
result_best['best_score'] = result_best[['lr_score', 'rf_score', 'svm_score']].idxmax(axis=1)
result_best = result_best[['party', 'best_score', 'lr_score', 'rf_score', 'svm_score']]
print("Average accuracy: ", avg_best_lr)                                               # print average best values
print("Average accuracy: ", avg_best_rf)                                               # print average best values
print("Average accuracy: ", avg_best_svm)                                              # print average best values
display(result_best)

Average accuracy:  0.7456245106521546
Average accuracy:  0.7543124566433861
Average accuracy:  0.6932356457353035


,party,best_score,lr_score,rf_score,svm_score
0,50PLUS,rf_score,0.802179,0.807130,0.790295
1,BBB,rf_score,0.837973,0.841314,0.830735
2,BIJ1,rf_score,0.782581,0.790968,0.758065
3,BVNL,rf_score,0.680046,0.692783,0.632960
4,CDA,lr_score,0.676357,0.676357,0.544816
5,CU,rf_score,0.687045,0.691412,0.571082
6,D66,rf_score,0.682744,0.695104,0.557925
7,DENK,rf_score,0.834264,0.840087,0.824800
8,FVD,rf_score,0.695194,0.714566,0.605934
9,GLPvdA,rf_score,0.821195,0.822889,0.801839


------------------------------------------------
<br>
<br>
<br>


## Final Models
--------------------------------------------

In [62]:
dur = time.time()
fm_results = pd.DataFrame(columns=['party','n_est', 'max_depth', 'min_split','fm_score'])
file_url  = file_path + '54_data_tuned_random_forest.csv' 
parameters = pd.read_csv(file_url)
parameters = parameters.drop(['option','rf_score'], axis=1)

# -------------------------------------------------------------------------------------~----------------------------------------
for party in tqdm(parties, desc = 'Party loop'):                                       # for each party
    
# load training and validation data 
# -------------------------------------------------------------------------------------~----------------------------------------
    X_trainvalid_url = file_path + f"40_models_imbalanced/" + f"{party}_X_trainvalid.csv"
    y_trainvalid_url = file_path + f"40_models_imbalanced/" + f"{party}_y_trainvalid.csv"
    X_trainvalid = pd.read_csv(X_trainvalid_url)                                       # read X_trainvalid
    X_trainvalid = X_trainvalid.drop(columns=['source', 'text','stemming_id','document_id'])
    y_trainvalid = pd.read_csv(y_trainvalid_url)                                       # read y_trainvalid
        
# load test data
# -------------------------------------------------------------------------------------~----------------------------------------
    X_test_url = file_path + f"40_models_imbalanced/" + f"{party}_X_test.csv"          # define X_test url
    y_test_url = file_path + f"40_models_imbalanced/" + f"{party}_y_test.csv"          # define y_test url
    X_test = pd.read_csv(X_test_url)                                                   # read X_test
    X_test = X_test.drop(columns=['source', 'text','stemming_id','document_id'])       # redefine X_test without columns
    y_test = pd.read_csv(y_test_url)                                                   # read y_test

# Define best model (parameters)
# -------------------------------------------------------------------------------------~----------------------------------------
    party_parameters = parameters[parameters['party'] == party].iloc[0]
    n_est = party_parameters['n_est']
    max_depth = party_parameters['max_depth']
    min_split = party_parameters['min_split']    

# Run best model 
# -------------------------------------------------------------------------------------~----------------------------------------
    # -----------------------------------------
    fm = RandomForestClassifier(n_estimators=n_est, max_depth=max_depth, min_samples_split=min_split, random_state=rs)
    fm.fit(X_trainvalid, y_trainvalid)                                                 # fit model
    fm_score = fm.score(X_test, y_test)                                                # score model
    fm_result = pd.DataFrame({  'party': [party],
                                'n_est': [n_est],
                                'max_depth': [max_depth],
                                'min_split': [min_split],
                                'fm_score': [fm_score]})                               # define result
        
    fm_results = pd.concat([fm_results, fm_result], ignore_index=True)                 # redefine results
    model_url = file_path + f"50_models/{party}.pkl"                                   # define model url
    joblib.dump(fm, model_url)                                                         # dump model

# -------------------------------------------------------------------------------------~----------------------------------------
display(fm_results)                                                                    # display results
avg_best_fm = fm_results['fm_score'].mean()                                            # define average best values
print("Average accuracy: ", avg_best_fm)                                               # print average best values
print('\n---------------------------------------------------------------------------------------------------------------------')
print(f"Code duration: {round((time.time()  - dur),3)} seconds")    

Party loop: 100%|██████████| 18/18 [58:51<00:00, 196.17s/it] 


,party,n_est,max_depth,min_split,fm_score
0,50PLUS,500,50,2,0.805150
1,CDA,250,25,5,0.696705
2,CU,500,25,5,0.700946
3,D66,500,50,5,0.691711
4,DENK,500,50,2,0.841786
5,FVD,500,25,2,0.706055
6,GLPvdA,250,50,5,0.824625
7,PVV,500,25,5,0.716606
8,PvdD,500,50,2,0.799418
9,SGP,500,25,5,0.684988


Average accuracy:  0.7543919702147022

---------------------------------------------------------------------------------------------------------------------
Code duration: 3531.127 seconds
